# Acoustic Scan 

In [1]:
# matching pursuit
# depth profiling
# attenuation with high f. reflection ok, transmission no
# look at acoustic resonances, dip in attenuation
# 

In [2]:
%load_ext autoreload
%autoreload 2
import numpy as np
from matplotlib import pyplot as plt
import sys

sys.path.append('..') # path to the src directory
sys.path.append('/home/xinqiao/new_mount/gaussian_sampler/ultrasonicTesting')
sys.path.append('/home/xinqiao/new_mount/gaussian_sampler/M3Learning-Util/src')
sys.path.append('/home/xinqiao/new_mount/gaussian_sampler/AutoPhysLearn/src')
sys.path.append('/home/xinqiao/new_mount/gaussian_sampler/Gaussian_Sampler/Gaussian_Sampler')


from scipy.signal import butter, sosfiltfilt
import copy
import math
import time
from tqdm import tqdm
import pickleJar as pj
import tomography as tm

In [3]:
from viz.visualize_scan_data import *
from IPython.display import display
import plotly.graph_objects as go

## Dataloader with preprocessing

In [4]:
from Gaussian_Sampler.data import datasets
from Gaussian_Sampler.data.datasets import morlet_1D_dataset_real

dset = morlet_1D_dataset_real(sq3lite_path='/home/xinqiao/new_mount/gaussian_sampler/ultrasound_data/sa-1-5a-1M-old-multiplex-scan-autorange-savegain.sqlite3',
                              dset_name='voltage_transmission_forward',
                              image_shape = (41,51),
                            #   crops = [(0,4000)] #(15000,19000)
                            ) 

sqliteToPickle Warning: pickle file /home/xinqiao/new_mount/gaussian_sampler/ultrasound_data/sa-1-5a-1M-old-multiplex-scan-autorange-savegain.pickle already exists. Conversion aborted.


/home/xinqiao/new_mount/gaussian_sampler/ultrasonicTesting/pickleJar.py:1185: RuntimeWarning: divide by zero encountered in log10
  logData = np.log10(abs(data))


In [5]:
# dset.display_dict_tree()

## Interactive Viewer with Slider

Use the slider below to browse through all scans interactively.

In [6]:
# # Create interactive viewer with slider (fast - uses ipywidgets)
# from Gaussian_Sampler.viz.visualize_scan_data import plotly_viewer
# viewer = plotly_viewer(dset)
# display(viewer)  # or just: viewer  (in Jupyter, the last line auto-displays)

## try training model on water with morlet packet

goals:
- figure out mean position and f of morlet packet
- using this, calculate speed of sound in this water

In [32]:
from Gaussian_Sampler.models.morlet_fitter import Fitter_AE, morlet_1D_fitters_real
from autophyslearn.spectroscopic.nn import block_factory, Conv_Block, FC_Block  # pyright: ignore[reportMissingImports]
from autophyslearn.spectroscopic.nn import Multiscale1DFitter
from Gaussian_Sampler.data.custom_sampler import Gaussian_Sampler
import torch

num_fits = 4 # number of curves to sum up
num_params = 4 # number of parameters to fit
# todo: change wandb naming to include noise level, group and regularization technique
# todo: test more num fits
model = Fitter_AE(function=morlet_1D_fitters_real,
                dset=dset,
                num_params=num_params,
                num_fits=num_fits,
                checkpoints_label='ultrasound_water',
                input_channels = 1,
                learning_rate=3e-6,
                device='cuda:0',
                encoder = Multiscale1DFitter,
                encoder_params = {
                    "model_block_dict": { # factory wrapper for blocks
                            "hidden_x1": block_factory(Conv_Block)(output_channels_list=[128,128], 
                                                                    kernel_size_list=[5,5], 
                                                                    pool_list=[2000,500], 
                                                                    max_pool=False),
                            # "hidden_xfc": block_factory(FC_Block)(output_size_list=[128,64]), # remove 2nd block and skip connections
                            # "hidden_x2": block_factory(Conv_Block)(output_channels_list=[32,16], 
                            #                                         kernel_size_list=[75,75], 
                            #                                         pool_list=[64,32], 
                            #                                         max_pool=True),
                            "hidden_embedding": block_factory(FC_Block)(output_size_list=[8*num_fits,num_params*num_fits], last=True),
                        },
                        # TEST: LIMITS,
                        # "skip_connections": {'hidden_xfc': 'hidden_embedding'},
                        "skip_connections": {},
                        "function_kwargs": {'limits': [1, # amplitude
                                                       dset.spec_len, # mean
                                                       dset.spec_len/10, # stdev
                                                       1e-2] # freq max (100 MHz)
                                            } 
                    },
                    # sampler = Gaussian_Sampler, # using random sampler
                    # sampler_params = {'dset': dset, 
                    #                     'batch_size': 100, 
                    #                     'gaussian_std': 3, 
                    #                     'orig_shape': dset.shape[0:-1], 
                    #                     'num_neighbors': 10, },
                )


### make graph for model


In [33]:
# nn.Tanh()

### Train model for several epochs


In [43]:
# import wandb
# wandb.init(group='sub_sampler_type', name='sub_noise_level') # later change config for regularization

model.train(epochs=5,save_every=500, batch_size=123, log_wandb=False, 
            # lr_scheduling=True,
            # coef1=1e-3
            )

/home/xinqiao/new_mount/gaussian_sampler/ultrasound_data/ultrasound_water/checkpoints/voltage_transmission_forward


100%|██████████| 17/17 [00:00<00:00, 28.22it/s]


Epoch: 000/005 | Train Loss: 0.0153
.............................


100%|██████████| 17/17 [00:00<00:00, 30.04it/s]


Epoch: 001/005 | Train Loss: 0.0124
.............................


...

Epoch: 002/005 | Train Loss: 0.0108
.............................


100%|██████████| 17/17 [00:00<00:00, 29.37it/s]


Epoch: 003/005 | Train Loss: 0.0097
.............................


100%|██████████| 17/17 [00:00<00:00, 29.38it/s]

Epoch: 004/005 | Train Loss: 0.0090
.............................


### Embeddings

In [44]:
def write_scaled_embedding(batch_size=1):
    for i, (idx, x) in enumerate(tqdm(model.dataloader, leave=True, total=len(model.dataloader))):
        with torch.no_grad():
            fits, params = model.encoder(x.float().to(model.device))
            fits = fits.cpu().numpy()
            params = params.cpu().numpy()
    return fits, params

fits, params = write_scaled_embedding(batch_size=1)

# sweep frequencies and search for resonances 
# attenuation in each layer accounts for spherical nature of wave
# data for one to 20 layers
# measure waveform from 30-50, measuring thermal gradient. shoul dbe the same as 40 (mean), so why isnt it?
# goal: use ultrasound to monitor the thermal expansion so we can decreasing charging rate. this way the battery is less likely to experience stress and can cycle more

  0%|          | 0/17 [00:00<?, ?it/s]

100%|██████████| 17/17 [00:00<00:00, 65.43it/s]


In [45]:
from Gaussian_Sampler.viz.visualize_scan_data import training_viewer

training_viewer(dset, fits, params)